### Notebook to create BLOCK-T422 for Camera Hexapod Full Array Mode Interpolation

This notebooks create the json block that goes one by one through all the Camera hexapod degrees of freedom and taking a triplet to check we are able to retrieve the wavefront across the field from the corners.

Created on: 2025-03-28

Author: Guillem Megias

In [ ]:
from lsst.ts.observing import ObservingBlock, ObservingScript 
from lsst.ts.aos.analysis import build_configuration_schema
import os
import numpy as np

In [ ]:
current_path = os.getcwd()
block_number = 'T422'
name = 'BLOCK-T422'
program = name
reason = "WET-009"
note = "full_array_mode_cam_"
labels = ['dz', 'dx', 'dy', 'rx', 'ry']
constraints = []

### Define configuration schema

In [ ]:
# Define the configurable properties that we will use in the configuration schema
properties = {
    "filter": {
        "description": "Filter to use.",
        "type": "string",
        "default": "r_57"
    },
    "day": {
        "description": "Day of the year for the reference state.",
        "type": "integer",
        "default": 1
    },
    "seq": {
        "description": "Sequence number for the reference state.",
        "type": "integer",
        "default": 1
    },
    "exp_time": {
        "description": "Exposure time.",
        "type": "number",
        "default": 30.0
    },
    "n_triplets": {
        "description": "Number of triplets for the closed loop.",
        "type": "integer",
        "default": 1
    },
    "dz": {
        "description": "z offset for the triplets.",
        "type": "number",
        "default": 1500.
    },
    "mode": {
        "description": "Mode of the aos sequence (TRIPLET, INTRA, EXTRA, PAIR).",
        "type": "string",
        "default": "TRIPLET"
    },
}

# Build the configuration schema for BLOCK-404
configuration_schema = build_configuration_schema(block_number, properties)
print(configuration_schema)

### Define scripts and block

In [ ]:
ranges = [100, 2000, 2000, 0.05, 0.05]
offset = 5
scripts = []

for idx, dof_range in enumerate(ranges):
    reset_state = ObservingScript(
        name="maintel/set_dof.py",
        standard=True,
        parameters= dict(
            day="$day",
            seqnum="$seqnum",
        )
    )

    dofs = np.zeros(50)
    dofs[idx + offset] = dof_range
    apply_state = ObservingScript(
        name="maintel/apply_dof.py",
        standard=True,
        parameters= dict(
            dofs=dofs.tolist(),
        )
    )

    triplet_script = ObservingScript(
        name="maintel/take_aos_sequence_lsstcam.py",
        standard=True,
        parameters= dict(
            filter="$filter",
            exposure_time="$exp_time",
            dz="$dz",
            n_sequences="$n_triplets",
            mode="$mode",
            program="$program",
            reason=reason,
            note=f"{note}{labels[idx]}"
        )
    )
    scripts.append(reset_state)
    scripts.append(apply_state)
    scripts.append(triplet_script)

scripts.append(reset_state)

In [ ]:
block = ObservingBlock(
    name = name,
    program = program,
    configuration_schema=configuration_schema,
    scripts = scripts,
)

### Save configurable block

In [ ]:
block.model_dump_json(indent=2)

output_file_path = f'{current_path}/aos/ts_config_ocs/Scheduler/observing_blocks_maintel/AOS/WEP/{program}.json'

with open(output_file_path, 'w') as file:
    file.write(block.model_dump_json(indent=2))